In [1]:
import pandas as pd
import numpy as np 
import os 
from pathlib import Path

## Caminhos locais

In [12]:
df = pd.read_csv('../data/metadata_final_cloud.csv')
df_filter = df[(df['dataset_id'] == 'jpr_dados') | (df['dataset_id'] == 'physionet') | (df['dataset_id'] == 'cq500_dados')]

In [ ]:


# Define base paths
jpr_base = Path(r"D:\Backup\Mestrado\dados\JPR\dataset_jpr_train\dataset_jpr_train")
physionet_base = Path(r"D:\Backup\Mestrado\dados\Physionet\physionet_scans\ct_scans")

# 1. Pre-index JPR directories (since they are split across folders 1, 2, 3)
jpr_paths = {}
for sub in ['1', '2', '3']:
    p = jpr_base / sub
    if p.exists():
        for folder in p.iterdir():
            if folder.is_dir():
                jpr_paths[folder.name] = folder.resolve().as_posix()

# 2. Function to resolve local path based on dataset_id
def get_local_path(row):
    pid = str(row['patient_id']).strip()
    dataset = row['dataset_id']
    
    if dataset == 'physionet':
        # Formats 49 -> 049.nii
        filename = f"{pid.zfill(3)}.nii"
        file_path = physionet_base / filename
        return file_path.resolve().as_posix() if file_path.exists() else None
        
    elif dataset == 'jpr_dados':
        # Formats 1171 -> 001171
        formatted_id = pid.zfill(6)
        return jpr_paths.get(formatted_id, None)
    
    return None

# 3. Apply to df_filter
df_filter['local_Path'] = df_filter.apply(get_local_path, axis=1)

# Check the result
print(df_filter[['dataset_id', 'patient_id', 'local_Path']].head())

In [ ]:
#df_filter.to_csv('../data/metadata_local.csv', index = False)

## 3D -> 2D -> save_path

In [2]:
df_metadata = pd.read_csv('../data/sampled_metadata_aux.csv')

df_metadata['local_Path'] = df_metadata['local_Path'].str.replace('D:/Backup/Mestrado/dados', '/home/jovyan/imgs')

df_metadata['local_Path'] = df_metadata['local_Path'].str.replace('D:/Backup/Mestrado/dados', '/home/jovyan/imgs')

In [3]:
#df_metadata[df_metadata['dataset_id'] == 'physionet'].iloc[1]

#df_nan = df_metadata[df_metadata["local_Path"].isna()]
#df_nan

In [6]:
df_metadata[df_metadata['dataset_id'] == 'physionet'].iloc[1].values

array([np.int64(50), nan, 'physionet', 'Female', np.float64(0.583), nan,
       np.float64(32.0), np.float64(nan), np.float64(0.3515625), '5.0',
       np.float64(1.0), np.float64(0.0), np.False_, np.float64(nan),
       np.float64(nan), np.float64(nan),
       '/home/validacao-retrospectiva-neuro-new/physionet_scans/ct_scans/050.nii',
       'Intraparenchymal/Fracture', np.float64(nan), np.float64(nan),
       np.float64(nan), np.float64(512.0), np.float64(512.0),
       np.float64(32.0), np.float64(nan),
       'D:/Backup/Mestrado/dados/Physionet/physionet_scans/ct_scans/050.nii',
       np.True_], dtype=object)

In [3]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from monai.data import ITKReader
from monai.transforms import (
    Compose,
    DeleteItemsd,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    Orientationd,
    Resized,
    ScaleIntensityRanged,
    Spacingd,
    ToDeviced,
)

# --- Diretórios de Saída ---
save_path_jpr = "/home/jovyan/imgs/png_data/jpr_dados"
save_path_physionet = "/home/jovyan/imgs/png_data/physionet"

# --- Pipeline de Transformações MONAI ---
deterministic_transforms = Compose(
    [
        LoadImaged(
            keys=["im"], image_only=False, reader=ITKReader(series_name="")
        ),
        EnsureTyped(keys=["im"]),
        EnsureChannelFirstd(keys=["im"]),
        ToDeviced(keys=["im"], device="cuda:0"),
        Orientationd(keys=["im"], axcodes="RAS"),
        ScaleIntensityRanged(
            keys=["im"],
            a_min=-15,
            a_max=85,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        Spacingd(
            keys=["im"],
            pixdim=(1.0, 1.0, 1.0),
            mode="bilinear",
        ),
        Resized(keys=["im"], spatial_size=(256, 256, -1)),
  # evita erro caso a chave não exista
    ]
)

# --- Processamento das Imagens ---
for _, row in tqdm(
    df_metadata.iterrows(),
    total=len(df_metadata),
    desc="Processando pacientes",
):
    patient_id = str(row["patient_id"])
    local_path = str(row["local_Path"])
    dataset_id = str(row["dataset_id"])

    # Define o diretório de destino de acordo com o dataset_id
    base_save_path = (
        save_path_jpr if dataset_id == "jpr_dados" else save_path_physionet
    )
    patient_dir = os.path.join(base_save_path, patient_id)
    os.makedirs(patient_dir, exist_ok=True)

    try:
        # Aplica o pipeline
        data_dict = {"im": local_path}
        transformed = deterministic_transforms(data_dict)

        # Converte o tensor para NumPy na CPU (formato C, H, W, D)
        vol = transformed["im"].detach().cpu().numpy()

        # Remove dimensão de canal se presente (ex: shape vira [H, W, D])
        if vol.ndim == 4:
            vol = vol[0]

        num_slices = vol.shape[-1]

        # Salva cada slice 2D em escala de cinza de 8 bits (0-255)
        for slice_idx in range(num_slices):
            slice_data = vol[:, :, slice_idx]

            # Converte float [0.0, 1.0] para uint8 [0, 255]
            slice_uint8 = (
                np.clip(slice_data, 0.0, 1.0) * 255.0
            ).astype(np.uint8)

            slice_img = Image.fromarray(slice_uint8)
            slice_img.save(
                os.path.join(patient_dir, f"slice_{slice_idx:04d}.png")
            )

    except Exception as e:
        print(f"Erro ao processar paciente {patient_id} ({local_path}): {e}")

mkdir -p failed for path /home/jovyan/.cache/matplotlib: [Errno 13] Permission denied: '/home/jovyan/.cache/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-4alx0681 because there was an issue with the default path (/home/jovyan/.cache/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Processando pacientes:   0%|                                                       | 0/813 [00:00<?, ?it/s]builtin type SwigPyPacked has no __module__ attribute
builtin type SwigPyObject has no __module__ attribute
builtin type swigvarlink has no __module__ attribute
Processando pacientes:   4%|█▊                                          | 34/813 [16:19<5:15:37, 24.31s/it]the directory: /home/jovyan/imgs/JPR/dataset_jpr_train/dataset_jpr_train/2/001554 contains more than one DICOM series.
ImageSeriesReader (0x6269c5e87850): Non uni

Erro ao processar paciente 370 (nan): applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x75cc0b958a50>


Processando pacientes:  18%|███████▎                                 | 146/813 [1:25:22<4:23:34, 23.71s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x6269c5e87850): Non uniform sampling or missing slices detected,  maximum nonuniformity:171.8

Processando pacientes:  22%|████████▊                                | 175/813 [1:49:59<4:15:10, 24.00s/it]

Erro ao processar paciente 2391 (nan): applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x75cc0b958a50>


Processando pacientes:  31%|████████████▌                            | 250/813 [2:33:40<4:03:55, 26.00s/it]

...

Processando pacientes:  65%|██████████████████████████▊              | 532/813 [6:59:09<1:51:14, 23.75s/it]

Erro ao processar paciente 2449 (nan): applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x75cc0b958a50>


Processando pacientes:  79%|████████████████████████████████▎        | 640/813 [7:40:18<1:05:27, 22.70s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x6269e165a390): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000972843

Processando pacientes:  82%|█████████████████████████████████▌       | 665/813 [7:50:02<1:03:49, 25.87s/it]

Erro ao processar paciente 2513 (nan): applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x75cc0b958a50>


Processando pacientes:  82%|███████████████████████████████████▍       | 670/813 [7:51:41<51:58, 21.80s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x6269e165a390): Non uniform sampling or missing slices detected,  maximum nonuniformity:169.388

Processando pacientes: 100%|███████████████████████████████████████████| 813/813 [8:42:00<00:00, 38.53s/it]


## Caminho por slice


In [10]:
import os 
import glob
df_train = pd.read_csv('/home/jovyan/work/data/splits/test_data.csv')






In [11]:
def get_slice_paths(folder_path):
    if not isinstance(folder_path, str) or not os.path.exists(folder_path):
        return []
    
    # Busca todos os arquivos .png (direto e recursivo)
    paths = glob.glob(os.path.join(folder_path, "*.png"))
    paths.extend(glob.glob(os.path.join(folder_path, "**", "*.png"), recursive=True))
    
    # Remove duplicatas e garante que são apenas arquivos reais
    return sorted(list(set(p for p in paths if os.path.isfile(p))))

# 1. Copia o DataFrame original
df_expanded = df_train.copy()

# 2. Gera a lista de fatias para cada diretório da coluna 'png_Path'
df_expanded["individual_path"] = df_expanded["png_Path"].apply(get_slice_paths)

# 3. Explode a coluna: cada elemento da lista vira uma nova linha,
# replicando todas as outras colunas existentes
df_expanded = df_expanded.explode("individual_path").reset_index(drop=True)

# 4. (Opcional) Remove eventuais pastas que não continham nenhum .png (valores NaN)
df_expanded = df_expanded.dropna(subset=["individual_path"]).reset_index(drop=True)

In [12]:
df_expanded.to_csv('/home/jovyan/work/data/splits/test_data_ind.csv', index = False)